Este projeto tem como objetivo prever a avaliação que um utilizador atribuiria a um determinado filme, com base no histórico de avaliações e nos géneros dos filmes do dataset MovieLens. Para isso, serão comparados os modelos de regressão **Ridge** e **Árvore de Decisão**.

### 1 - Carregamento e Exploração de Dados

In [1]:
import pandas as pd

ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

display(ratings.head())
display(movies.head())

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [2]:
print("Número de avaliações:", len(ratings))
print("Número de utilizadores:", ratings["userId"].nunique())
print("Número de filmes no catálogo:", len(movies))

Número de avaliações: 100836
Número de utilizadores: 610
Número de filmes no catálogo: 9742


In [3]:
print("Valores em falta nas avaliações:\n")
print(ratings.isna().sum())

Valores em falta nas avaliações:

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


### 2 - Junção das Avaliações aos Títulos e Géneros dos Filmes

In [4]:
avaliacoes = ratings.merge(
    movies,
    on="movieId",
    how="left",
    validate="many_to_one"
)

display(avaliacoes[["userId", "title", "genres", "rating"]].head(10))

,userId,title,genres,rating
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,4.0
1,1,Grumpier Old Men (1995),Comedy|Romance,4.0
2,1,Heat (1995),Action|Crime|Thriller,4.0
3,1,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,5.0
4,1,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,5.0
5,1,From Dusk Till Dawn (1996),Action|Comedy|Horror|Thriller,3.0
6,1,Bottle Rocket (1996),Adventure|Comedy|Crime|Romance,5.0
7,1,Braveheart (1995),Action|Drama|War,4.0
8,1,Rob Roy (1995),Action|Drama|Romance|War,5.0
9,1,Canadian Bacon (1995),Comedy|War,5.0


Apresentam-se as primeiras dez avaliações, associadas aos respetivos títulos e géneros dos filmes. Neste conjunto de dados, estas linhas correspondem ao utilizador 1.

**Observação**: o `.head(x)` vai buscar os `x` primeiros registos do DataFrame Pandas.

*Analisando cenários:*
Caso se queira explorar os filmes avaliados pelo `userId` 1:

In [5]:
avaliacoes_user1 = avaliacoes.loc[
    avaliacoes["userId"] == 1,
    ["title", "genres", "rating"]
]

print("Número de filmes avaliados pelo utilizador 1:", len(avaliacoes_user1))
display(avaliacoes_user1.head(10))

Número de filmes avaliados pelo utilizador 1: 232


,title,genres,rating
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,4.0
1,Grumpier Old Men (1995),Comedy|Romance,4.0
2,Heat (1995),Action|Crime|Thriller,4.0
3,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,5.0
4,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,5.0
5,From Dusk Till Dawn (1996),Action|Comedy|Horror|Thriller,3.0
6,Bottle Rocket (1996),Adventure|Comedy|Crime|Romance,5.0
7,Braveheart (1995),Action|Drama|War,4.0
8,Rob Roy (1995),Action|Drama|Romance|War,5.0
9,Canadian Bacon (1995),Comedy|War,5.0


### 3 - Divisão Treino e Teste
O nosso objetivo é: dado um utilizador, um filme e os seus géneros, prever a nota atribuída. Serão utilizados dois modelos de Regressão: **Ridge** e **Árvore de Decisão**.

In [6]:
from sklearn.model_selection import train_test_split

# Selecionamos identificadores e géneros; convertemos os IDs para string para serem tratados como categorias.
X = avaliacoes[["userId", "movieId", "genres"]].copy()
X[["userId", "movieId"]] = X[["userId", "movieId"]].astype(str)

# Variável alvo que queremos prever
y = avaliacoes["rating"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Avaliações de treino:", len(X_treino))
print("Avaliações de teste:", len(X_teste))

Avaliações de treino: 80668
Avaliações de teste: 20168


### 4 - Preparação das Categorias e Criação dos Modelos

As variáveis de entrada incluem os identificadores do utilizador (*userId*) e do filme (*movieId*), bem como os géneros cinematográficos (*genres*).

Apesar de os identificadores serem números, representam categorias nominais e não quantidades. Por isso, tanto os identificadores como os géneros serão transformados em colunas binárias (0 ou 1) antes de serem utilizados pelos modelos.

**One-Hot Encoding e Codificação de Géneros**

As categorias passam a ser representadas por colunas de "sim" (`1`) ou "não" (`0`).

Exemplo:

*Antes*:

| userId | movieId | genres |
|---|---|---|
| 1 | 10 | Action\|Thriller |
| 2 | 20 | Comedy |

*Depois da codificação binária*:

| É o utilizador 1? | É o utilizador 2? | É o filme 10? | É o filme 20? | Género Action? | Género Comedy? | Género Thriller? |
|---|---|---|---|---|---|---|
| 1 | 0 | 1 | 0 | 1 | 0 | 1 |
| 0 | 1 | 0 | 1 | 0 | 1 | 0 |

*Exemplos ilustrativos, apenas.*

Isto evita interpretar identificadores como quantidades e permite que modelos baseados em árvores criem regras por género cinematográfico.

### 5 - Uso do OneHotEncoder e ColumnTransformer

In [7]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.compose import ColumnTransformer

exemplo = X_treino[["userId", "movieId", "genres"]].head(5).copy()

codificador_exemplo = ColumnTransformer([
    ("ids", OneHotEncoder(handle_unknown="ignore"), ["userId", "movieId"]),
    ("generos", CountVectorizer(token_pattern=r"[^|]+", lowercase=False), "genres")
])

exemplo_transformado = codificador_exemplo.fit_transform(exemplo)

tabela_exemplo = pd.DataFrame(
    exemplo_transformado.toarray(),
    columns=codificador_exemplo.get_feature_names_out(),
    index=exemplo.index
)

print("Dados antes:")
display(exemplo)

print("Com One-Hot Encoding + Géneros:")
display(tabela_exemplo)

Dados antes:


,userId,movieId,genres
80568,509,7347,Mystery|Thriller
50582,326,71462,Documentary
8344,57,2115,Action|Adventure|Fantasy
99603,610,1127,Action|Adventure|Sci-Fi|Thriller
71701,462,2409,Action|Drama


Com One-Hot Encoding + Géneros:


,ids__userId_326,ids__userId_462,ids__userId_509,ids__userId_57,ids__userId_610,ids__movieId_1127,ids__movieId_2115,ids__movieId_2409,ids__movieId_71462,ids__movieId_7347,generos__Action,generos__Adventure,generos__Documentary,generos__Drama,generos__Fantasy,generos__Mystery,generos__Sci-Fi,generos__Thriller
80568,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
50582,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
8344,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
99603,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0
71701,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


### 5.1 - Pipeline de Regressão **Ridge**

A **Regressão Ridge** é uma regressão linear com regularização L2. Além de procurar reduzir o erro das previsões, controla o sobreajuste (*overfitting*) ao penalizar coeficientes muito elevados.

O modelo representa a tendência de um utilizador atribuir notas mais altas, o prestígio próprio de cada filme e o peso médio dos seus géneros. O parâmetro **alpha** controla a intensidade da regularização e será otimizado posteriormente.

#### Nota prevista = intercepto + efeito do utilizador + efeito do filme + efeito dos géneros

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

preprocessador = ColumnTransformer([
    ("ids", OneHotEncoder(handle_unknown="ignore"), ["userId", "movieId"]),
    ("generos", CountVectorizer(token_pattern=r"[^|]+", lowercase=False), "genres")
])

modelo_ridge = Pipeline([
    ("categorias", preprocessador),
    (
        "regressao",
        Ridge(
            alpha=1.0,     # valor inicial de regularização
            solver="lsqr"  # método iterativo rápido para matrizes esparsas
        )
    )
])

display(modelo_ridge)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('categorias', ...), ('regressao', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ids', ...), ('generos', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3

### 5.2 - Pipeline Árvore de Decisão

A **Árvore de Decisão** divide os dados através de regras sucessivas com o objetivo de formar grupos com avaliações semelhantes.

Neste projeto, as regras serão construídas a partir dos géneros cinematográficos e dos identificadores codificados dos utilizadores e dos filmes. A profundidade máxima (`max_depth`) e o número mínimo de exemplos por folha (`min_samples_leaf`) permitem controlar a complexidade da árvore e serão ajustados na fase de otimização.

In [9]:
from sklearn.tree import DecisionTreeRegressor

preprocessador_arvore = ColumnTransformer([
    ("ids", OneHotEncoder(handle_unknown="ignore"), ["userId", "movieId"]),
    ("generos", CountVectorizer(token_pattern=r"[^|]+", lowercase=False), "genres")
])

modelo_arvore = Pipeline([
    ("categorias", preprocessador_arvore),
    (
        "regressao",
        DecisionTreeRegressor(
            max_depth=10,        # profundidade máxima inicial da árvore
            min_samples_leaf=20, # exige pelo menos 20 avaliações de treino em cada folha
            random_state=42
        )
    )
])

display(modelo_arvore)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('categorias', ...), ('regressao', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ids', ...), ('generos', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3

### 5.3 - Uso das Pipelines

Cada modelo foi incluído numa pipeline que reúne a codificação das categorias e a regressão.

As pipelines aprenderão as categorias apenas com a parte usada para treino em cada divisão. A parte da **validação** aplicará essa mesma codificação sem participar na *aprendizagem*. Essa organização mantém o mesmo procedimento de preparação durante o treino e a previsão e evita fuga de dados (*data leakage*) entre o *treino* e a *validação*.

### 6 - Validação e Otimização dos Modelos

Para comparar os modelos e escolher os seus hiperparâmetros, será utilizada a validação cruzada com três divisões (*3-Fold Cross-Validation*) no conjunto de treino.

Em cada avaliação, duas partes serão utilizadas para *treinar* e a outra para *validar*. O processo repete-se até que todas as partes tenham passado uma vez pela validação.

In [10]:
from sklearn.model_selection import KFold, GridSearchCV

# KFold divide o conjunto de treino em K subconjuntos de tamanho igual para validação cruzada.
validacao = KFold(
    n_splits=3,      # treino dividido em três partes
    shuffle=True,    # baralha as linhas antes de formar as partes
    random_state=42  # garante a reprodutibilidade das divisões
)

### 6.1 - Otimização do Modelo Ridge

Será otimizado o parâmetro *alpha*, que regulariza os coeficientes das variáveis.

Valores maiores penalizam mais os coeficientes, restringindo o modelo. Esse tipo de restrição reduz o sobreajuste (*overfitting*), mas se for excessivo pode levar ao subajuste (*underfitting*). A validação cruzada permite selecionar o valor com menor erro entre os valores experimentados.

#### GridSearchCV

O **GridSearchCV** testa exaustivamente os valores candidatos do hiperparâmetro *alpha* usando validação cruzada (*CV: Cross-Validation*).

In [11]:
# O parâmetro otimizado será o alpha do modelo Ridge
parametros_ridge = {
    "regressao__alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
}

busca_ridge = GridSearchCV(
    estimator=modelo_ridge,
    param_grid=parametros_ridge,
    scoring="neg_mean_squared_error",
    cv=validacao,
    n_jobs=-1
)

busca_ridge.fit(X_treino, y_treino)

print("Melhores Parâmetros:", busca_ridge.best_params_)

rmse_validacao_ridge = (-busca_ridge.best_score_) ** 0.5
print(f"RMSE de validação: {rmse_validacao_ridge:.3f}")

Melhores Parâmetros: {'regressao__alpha': 10.0}
RMSE de validação: 0.871


### 6.2 - Otimização da Árvore de Decisão

Na Árvore de Decisão, serão otimizados a profundidade máxima (*max_depth*) e o número mínimo de avaliações por folha (*min_samples_leaf*):

* *max_depth*: limita a profundidade da árvore (quantos níveis de decisão consecutivos pode ter).
* *min_samples_leaf*: define quantas avaliações de treino devem existir, no mínimo, em cada folha final.

Serão testadas profundidades máximas de 10 e 20, bem como mínimos de 20 e 100 avaliações por folha.

In [12]:
parametros_arvore = {
    "regressao__max_depth": [10, 20],
    "regressao__min_samples_leaf": [20, 100]
}

busca_arvore = GridSearchCV(
    estimator=modelo_arvore,
    param_grid=parametros_arvore,
    scoring="neg_mean_squared_error",
    cv=validacao,
    n_jobs=-1
)

busca_arvore.fit(X_treino, y_treino)

print("Melhores Parâmetros:", busca_arvore.best_params_)

rmse_validacao_arvore = (-busca_arvore.best_score_) ** 0.5
print(f"RMSE de validação: {rmse_validacao_arvore:.3f}")

Melhores Parâmetros: {'regressao__max_depth': 20, 'regressao__min_samples_leaf': 20}
RMSE de validação: 0.970


### 6.3 - Comparação dos Resultados

In [13]:
comparacao_validacao = pd.DataFrame([
    {
        "Modelo": "Ridge",
        "RMSE de Validação": rmse_validacao_ridge,
        "Melhores Parâmetros": busca_ridge.best_params_
    },
    {
        "Modelo": "Árvore de Decisão",
        "RMSE de Validação": rmse_validacao_arvore,
        "Melhores Parâmetros": busca_arvore.best_params_
    }
]).sort_values("RMSE de Validação").reset_index(drop=True)

display(comparacao_validacao)

,Modelo,RMSE de Validação,Melhores Parâmetros
0,Ridge,0.871263,{'regressao__alpha': 10.0}
1,Árvore de Decisão,0.970009,"{'regressao__max_depth': 20, 'regressao__min_s..."


Pelos resultados, o **Modelo Ridge** apresenta o menor erro de validação (*RMSE*), sendo selecionado para a avaliação final no conjunto de teste.

O RMSE está expresso na escala das avaliações (em estrelas) e penaliza mais os erros elevados. Não representa uma percentagem de acerto.

### 7 - Avaliação Final no Conjunto de Teste

Após a seleção do modelo através da validação cruzada, será avaliado o seu desempenho nas 20.168 avaliações do conjunto de teste, que não participaram no treino nem na escolha dos hiperparâmetros.

Serão calculados o **MAE** e o **RMSE**, comparando tanto a **Ridge** como a **Árvore de Decisão** otimizadas face a uma referência simples que prevê sempre a média das avaliações do conjunto de treino.

In [14]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

melhor_modelo_ridge = busca_ridge.best_estimator_
melhor_modelo_arvore = busca_arvore.best_estimator_

previsoes_ridge = melhor_modelo_ridge.predict(X_teste)
previsoes_arvore = melhor_modelo_arvore.predict(X_teste)

media_treino = y_treino.mean()
previsoes_referencia = np.full(len(y_teste), media_treino)

print(f"Média das Avaliações de Treino: {media_treino:.3f}")
print("Número de Previsões no Teste:", len(previsoes_ridge))

Média das Avaliações de Treino: 3.503
Número de Previsões no Teste: 20168


### 7.1 - Cálculo das Métricas

In [15]:
resultados_teste = []

for nome, previsoes in [
    ("Referência: Média do Treino", previsoes_referencia),
    ("Árvore de Decisão Otimizada", previsoes_arvore),
    ("Ridge Otimizada", previsoes_ridge)
]:
    mae = mean_absolute_error(y_teste, previsoes)
    rmse = np.sqrt(mean_squared_error(y_teste, previsoes))

    resultados_teste.append({
        "Modelo": nome,
        "MAE do teste": mae,
        "RMSE do teste": rmse
    })

tabela_resultados_teste = pd.DataFrame(resultados_teste)
display(tabela_resultados_teste.round(4))

,Modelo,MAE do teste,RMSE do teste
0,Referência: Média do Treino,0.8316,1.0488
1,Árvore de Decisão Otimizada,0.7711,0.9800
2,Ridge Otimizada,0.6688,0.8730


O **MAE** corresponde à média dos erros absolutos entre as notas reais e previstas (por exemplo, um MAE de 0,67 significa que o modelo erra, em média, 0,67 estrelas em valor absoluto). O **RMSE** corresponde à raiz do erro quadrático médio, atribuindo maior peso aos erros grandes.

Para ambas as métricas, valores menores indicam melhores previsões.

### 7.2 - Notas Reais e Previstas

In [16]:
# Preservar a ordem das linhas usada na previsão.
exemplos_teste = X_teste.copy()

exemplos_teste["Nota real"] = y_teste.to_numpy()
exemplos_teste["Nota prevista"] = previsoes_ridge
exemplos_teste["Erro absoluto"] = (
    exemplos_teste["Nota real"] - exemplos_teste["Nota prevista"]
).abs()

# Compatibilizar o tipo do identificador para juntar os títulos.
titulos = movies[["movieId", "title"]].copy()
titulos["movieId"] = titulos["movieId"].astype(str)

exemplos_teste = exemplos_teste.merge(
    titulos,
    on="movieId",
    how="left",
    validate="many_to_one"
)

display(
    exemplos_teste[
        [
            "userId",
            "title",
            "genres",
            "Nota real",
            "Nota prevista",
            "Erro absoluto"
        ]
    ].head(10).round(3)
)

,userId,title,genres,Nota real,Nota prevista,Erro absoluto
0,432,Robin Hood (2010),Action|Adventure|Drama|Romance|War,4.5,3.394,1.106
1,288,In the Line of Fire (1993),Action|Thriller,3.0,3.322,0.322
2,599,Point Break (1991),Action|Crime|Thriller,3.0,2.493,0.507
3,42,Who Framed Roger Rabbit? (1988),Adventure|Animation|Children|Comedy|Crime|Fant...,4.0,3.791,0.209
4,75,"Hunt for Red October, The (1990)",Action|Adventure|Thriller,4.0,3.403,0.597
5,51,Lord of Illusions (1995),Horror,4.0,3.253,0.747
6,354,300 (2007),Action|Fantasy|War|IMAX,3.5,3.812,0.312
7,416,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,4.5,3.612,0.888
8,438,Charlie's Angels: Full Throttle (2003),Action|Adventure|Comedy|Crime|Thriller,0.5,2.815,2.315
9,73,Anchorman: The Legend of Ron Burgundy (2004),Comedy,3.5,3.831,0.331
